# Lab 10 — Part A: Lakehouse Federation
Querying an external PostgreSQL database (Neon) in place via a Unity Catalog connection and foreign catalog.

## Connection

In [0]:
CREATE CONNECTION IF NOT EXISTS natalia_neon_conn
TYPE postgresql
OPTIONS(
    host 'ep-red-star-b9gfmatw.c-7.eu-central-1.aws.neon.tech',
    port '5432',
  user 'neondb_owner',
  password secret('lab10', 'neon-password')
);

In [0]:
DESCRIBE CONNECTION natalia_neon_conn;

## Foreign catalog

In [0]:
CREATE FOREIGN CATALOG IF NOT EXISTS natalia_neon_catalog
USING CONNECTION natalia_neon_conn
OPTIONS (database 'neondb');

In [0]:
SHOW SCHEMAS IN natalia_neon_catalog;

In [0]:
SHOW TABLES IN natalia_neon_catalog.sales;

## Querying external tables via the foreign catalog

In [0]:
SELECT *
FROM natalia_neon_catalog.sales.customers
ORDER BY customer_id
LIMIT 10;

In [0]:
DESCRIBE TABLE natalia_neon_catalog.sales.orders;

In [0]:
DESCRIBE TABLE natalia_neon_catalog.sales.customers;

In [0]:
SELECT
  status,
  count(*)              AS orders_cnt,
  round(sum(amount), 2) AS revenue
FROM natalia_neon_catalog.sales.orders
WHERE order_date >= '2025-12-01'
GROUP BY status
ORDER BY revenue DESC;

In [0]:
SELECT count(*) AS orders_before_insert
FROM natalia_neon_catalog.sales.orders;

In [0]:
SELECT count(*) AS orders_after_insert
FROM natalia_neon_catalog.sales.orders;

## Pushdown

In [0]:
EXPLAIN FORMATTED
SELECT
  status,
  count(*)              AS orders_cnt,
  round(sum(amount), 2) AS revenue
FROM natalia_neon_catalog.sales.orders
WHERE order_date >= '2025-12-01'
GROUP BY status
ORDER BY revenue DESC;

In [0]:
EXPLAIN FORMATTED
SELECT *
FROM natalia_neon_catalog.sales.orders;

## Federation vs ingestion
A physical Delta copy of `orders` is created (ingestion) to compare with querying Neon in place (federation).

In [0]:
USE CATALOG dbr_dev_ua5816bd;
CREATE SCHEMA IF NOT EXISTS natalia_lab10;
USE SCHEMA natalia_lab10;

In [0]:
CREATE OR REPLACE TABLE orders_delta AS
SELECT
  *,
  current_timestamp() AS _ingested_at
FROM natalia_neon_catalog.sales.orders;

In [0]:
SELECT count(*) AS rows_in_delta, max(_ingested_at) AS ingested_at
FROM orders_delta;

### Same query: federation vs Delta copy

In [0]:
SELECT status, count(*) AS orders_cnt, round(sum(amount), 2) AS revenue
FROM natalia_neon_catalog.sales.orders
WHERE order_date >= '2025-12-01'
GROUP BY status
ORDER BY revenue DESC;

In [0]:
SELECT status, count(*) AS orders_cnt, round(sum(amount), 2) AS revenue
FROM orders_delta
WHERE order_date >= '2025-12-01'
GROUP BY status
ORDER BY revenue DESC;

### Freshness: a new order inserted in Neon after ingestion


In [0]:
SELECT 'federation (Neon, live)' AS source, count(*) AS orders_cnt
FROM natalia_neon_catalog.sales.orders
UNION ALL
SELECT 'delta copy (snapshot)', count(*)
FROM orders_delta;

## Joining an external table with a Delta table


In [0]:
SELECT
  c.city,
  c.segment,
  count(*)                AS orders_cnt,
  round(sum(o.amount), 2) AS revenue
FROM orders_delta AS o
JOIN natalia_neon_catalog.sales.customers AS c
  ON o.customer_id = c.customer_id
WHERE o.order_date >= '2025-12-01'
GROUP BY c.city, c.segment
ORDER BY revenue DESC;